In [1]:
# 安装 ffmpeg (用于提取音频)
!conda install -c conda-forge ffmpeg -y
# 安装 openai-whisper (语音转文字)
!pip install openai-whisper

Channels:
 - conda-forge
Platform: linux-64
Solving environment: done

# All requested packages already installed.



In [2]:
import os
import zipfile
import subprocess
import time  # 新增：用于等待
import vertexai
from vertexai.generative_models import GenerativeModel, Part
from google.cloud import storage
from google.api_core import exceptions  # 新增：用于捕获Google API特定错误
import shutil

# 1. 环境初始化
PROJECT_ID = "gen-lang-client-0371685655"
LOCATION = "us-central1"
vertexai.init(project=PROJECT_ID, location=LOCATION)

MODEL_NAME = "gemini-2.5-pro" 
# 注意：2.5-pro 目前可能是 preview 或非公开，如果报错请切回 gemini-1.5-pro-002
model = GenerativeModel(MODEL_NAME)

# 2. 路径配置
BUCKET_NAME = 'xhs-humor-data'
SEARCH_KEYWORD = "十万个梗库"
LOCAL_BASE_DIR = './raw_videos_temp'
RESULT_BASE_DIR = './meme_results'
CLOUD_OUTPUT_DIR = "extracted_knowledge" # 定义云端输出目录变量

os.makedirs(LOCAL_BASE_DIR, exist_ok=True)
os.makedirs(RESULT_BASE_DIR, exist_ok=True)

storage_client = storage.Client()
bucket = storage_client.bucket(BUCKET_NAME)

# --- [改进点 2]：预加载已完成列表（断点续传核心）---
print("正在同步云端已完成的任务列表，以便实现断点续传...")
# 获取云端所有已经存在的 txt 文件路径，存入集合（Set）中，查询速度极快 O(1)
existing_blobs = set([b.name for b in bucket.list_blobs(prefix=CLOUD_OUTPUT_DIR)])
print(f"云端已存在 {len(existing_blobs)} 个解析结果，这些将被跳过。\n")
# --------------------------------------------------

# 3. 获取目标文件并预扫描总量
all_blobs = storage_client.list_blobs(BUCKET_NAME)
target_zips = [b for b in all_blobs if SEARCH_KEYWORD in b.name and b.name.endswith('.zip')]

print(f"正在预扫描视频总量以计算全局进度...")
total_videos = 0
task_list = []

for zip_blob in target_zips:
    temp_zip = os.path.join(LOCAL_BASE_DIR, "scan.zip")
    zip_blob.download_to_filename(temp_zip)
    with zipfile.ZipFile(temp_zip, 'r') as z:
        v_list = [f for f in z.namelist() if f.lower().endswith(('.mp4', '.mov', '.avi')) and not f.startswith('__MACOSX')]
        total_videos += len(v_list)
        task_list.append((zip_blob, v_list))
    os.remove(temp_zip)

print(f"共计待处理视频总数: {total_videos}\n")

# --- [改进点 1]：定义带重试机制的请求函数 ---
def generate_with_retry(prompt_text, audio_bytes, max_retries=5):
    """
    尝试调用 Gemini API。
    如果遇到 429 错误，通过指数退避策略进行等待并重试。
    """
    retry_count = 0
    wait_time = 10  # 初始等待 10 秒
    
    while retry_count < max_retries:
        try:
            audio_part = Part.from_data(data=audio_bytes, mime_type="audio/mpeg")
            # 这里的 request 不需要传 prompt 列表，直接组合即可
            response = model.generate_content([prompt_text, audio_part])
            return response.text.strip()
        
        except exceptions.ResourceExhausted:
            # 捕获 429 错误
            print(f"  [!] 触发限流 (429)，等待 {wait_time} 秒后重试 (第 {retry_count+1}/{max_retries} 次)...")
            time.sleep(wait_time)
            retry_count += 1
            wait_time *= 2  # 指数增加等待时间：10s -> 20s -> 40s...
            
        except exceptions.InternalServerError:
             # 偶尔 Google 服务器内部错误，也可以简单重试
            print(f"  [!] 服务器内部错误 (500)，等待 5 秒重试...")
            time.sleep(5)
            retry_count += 1
            
        except Exception as e:
            # 其他错误直接抛出，不重试
            raise e
            
    raise Exception("重试次数耗尽，API 调用失败")
# --------------------------------------------------

# 4. 核心处理循环
processed_count = 0
skipped_count = 0 # 统计跳过数量

for zip_blob, video_files in task_list:
    subject_name = zip_blob.name.split('/')[-1].replace('.zip', '')
    subject_dir = os.path.join(RESULT_BASE_DIR, subject_name)
    os.makedirs(subject_dir, exist_ok=True)
    
    # 下载并解压当前主题包
    local_zip = os.path.join(LOCAL_BASE_DIR, "current.zip")
    print(f"\n>>> 正在下载并解压主题: {subject_name}")
    zip_blob.download_to_filename(local_zip)
    
    try:
        with zipfile.ZipFile(local_zip, 'r') as z:
            z.extractall(subject_dir)
    except zipfile.BadZipFile:
        print(f"错误: 压缩包损坏 {subject_name}，跳过")
        continue
    
    # 遍历处理该主题下的每一个视频
    for root, _, files in os.walk(subject_dir):
        for file in files:
            if file.lower().endswith(('.mp4', '.mov', '.avi')) and not file.startswith('._'):
                processed_count += 1
                video_path = os.path.join(root, file)
                audio_path = os.path.join(root, os.path.splitext(file)[0] + ".mp3")
                txt_name = f"{os.path.splitext(file)[0]}.txt"
                
                # 定义云端目标路径
                cloud_txt_path = f"{CLOUD_OUTPUT_DIR}/{subject_name}/{txt_name}"

                # --- [改进点 2]：断点续传检测 ---
                if cloud_txt_path in existing_blobs:
                    print(f"[{processed_count}/{total_videos}] 跳过已存在: {file}")
                    skipped_count += 1
                    # 清理本地视频防止占空间
                    if os.path.exists(video_path): os.remove(video_path)
                    continue
                # ------------------------------
                
                # 计算全局进度
                percent = (processed_count / total_videos) * 100
                print(f"进度: [{percent:.2f}%] | 正在处理: {file}")
                
                try:
                    # 第一步：即时提取音频
                    subprocess.run([
                        'ffmpeg', '-i', video_path, 
                        '-vn', '-acodec', 'libmp3lame', '-q:a', '2', 
                        audio_path
                    ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
                    
                    # 第二步：将音频直接喂给 Gemini (使用带重试的函数)
                    prompt = """
                    作为互联网梗百科专家，请听这段音频并提取：
                    1. 【梗名称】：
                    2. 【梗来源】：（原始出处/背景）
                    3. 【梗含义】：（为什么好笑/逻辑）
                    4. 【用法建议】：（适用文案场景）
                    直接输出以上结构化内容。
                    """
                    
                    with open(audio_path, "rb") as f_audio:
                        audio_data = f_audio.read()
                    
                    # --- 调用重试函数 ---
                    result_text = generate_with_retry(prompt, audio_data)
                    # -------------------
                    
                    # 第三步：生成 TXT 并即时上传云端
                    txt_local_path = os.path.join(subject_dir, txt_name)
                    with open(txt_local_path, "w", encoding="utf-8") as f_txt:
                        f_txt.write(result_text)
                    
                    # 上传云端
                    bucket.blob(cloud_txt_path).upload_from_filename(txt_local_path)
                    
                    # 更新内存中的“已完成列表”，防止本轮重复（虽然不太可能）
                    existing_blobs.add(cloud_txt_path)
                    
                    # 第四步：立即清理本地重文件
                    if os.path.exists(video_path): os.remove(video_path)
                    if os.path.exists(audio_path): os.remove(audio_path)
                    
                except Exception as e:
                    print(f"处理文件 {file} 失败: {e}")
                    # 即使失败，也要尝试清理本地临时文件，防止堆积
                    if os.path.exists(audio_path): os.remove(audio_path)

    # 清理当前压缩包缓存
    if os.path.exists(local_zip):
        os.remove(local_zip)
    
    # 清理当前解压目录（可选：如果你想彻底省空间，处理完一个 zip 就删掉解压文件夹）
    shutil.rmtree(subject_dir) 

print(f"\n--- 任务全部完成！共扫描 {processed_count}，实际处理 {processed_count - skipped_count}，跳过 {skipped_count} ---")

/opt/conda/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.cloud.aiplatform_v1beta1 once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.cloud.aiplatform_v1beta1 past that date.
  warnings.warn(message, FutureWarning)
/opt/conda/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.cloud.aiplatform_v1 once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.cloud.aiplatform_v1 past that date.
  warnings.warn(message, FutureWarning)
/opt/conda/lib/python3.10/site-packages/google/api_core/_python_version_supp

正在同步云端已完成的任务列表，以便实现断点续传...


/opt/conda/lib/python3.10/site-packages/vertexai/generative_models/_generative_models.py:433: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


云端已存在 298 个解析结果，这些将被跳过。

正在预扫描视频总量以计算全局进度...
共计待处理视频总数: 521


>>> 正在下载并解压主题: 小红书博主【十万个梗库】笔记批量下载_20260210122917
[1/521] 跳过已存在: 十万个梗库之：人类好可爱啊 #人类迷惑行_1.mp4
[2/521] 跳过已存在: 十万个梗库之：当工具做了噩梦 #抽象 #_1.mp4
[3/521] 跳过已存在: 十万个大事件之：短视频减速带 #影视 #_1.mp4
[4/521] 跳过已存在: 十万个梗库之：野猪佩奇 #佩奇 #抽象 _1.mp4
[5/521] 跳过已存在: 十万个大事件之：那些神级群星合唱 #群星_1.mp4
[6/521] 跳过已存在: 十万个大事件之：川渝人均古风 #川渝_1.mp4
[7/521] 跳过已存在: 十万个大事件之：宗门大会 #蔡徐坤 #表_1.mp4
[8/521] 跳过已存在: 十万个梗库之：原神nb #原神 #抽象 _1.mp4
[9/521] 跳过已存在: 十万个梗库之：网络音乐五巨头 #网络歌手_1.mp4
[10/521] 跳过已存在: 十万个大事件之：登录即领 #新年 #爱自_1.mp4
[11/521] 跳过已存在: 十万个梗库之：对齐颗粒度 #打工人 #职_1.mp4
[12/521] 跳过已存在: 十万个梗库之：等等党 #数码 #内存 #_1.mp4
[13/521] 跳过已存在: 十万个梗库之：四川婆罗门口音 #方言 #_1.mp4
[14/521] 跳过已存在: _1.mp4
[15/521] 跳过已存在: 十万个大事件之：逆转时间的公式 #过年_1.mp4
[16/521] 跳过已存在: 十万个梗库之：诈片新曲 #瑞克摇 #表情_1.mp4
[17/521] 跳过已存在: 十万个大事件：给大家丢个净化 #净化_1.mp4
[18/521] 跳过已存在: 十万个梗库之：次世代老登 #游戏_1.mp4
[19/521] 跳过已存在: 十万个大事件之：正视自己的年纪 #老登_1.mp4
[20/521] 跳过已存在: 十万个梗库之：魔性舞蹈鸽表情包 #表情包_1.mp4

>>> 正在下载并解压主题: 小红书博主【十万个梗库】笔记批量下载_20260210123101
[21/521] 跳过已存在: 十万

In [3]:
!gsutil -m cp -r gs://xhs-humor-data/extracted_knowledge .

Copying gs://xhs-humor-data/extracted_knowledge/小红书博主【十万个梗库】笔记批量下载_20260210122917/_1.txt...
Copying gs://xhs-humor-data/extracted_knowledge/小红书博主【十万个梗库】笔记批量下载_20260210122917/十万个梗库之：当工具做了噩梦 #抽象 #_1.txt...
Copying gs://xhs-humor-data/extracted_knowledge/小红书博主【十万个梗库】笔记批量下载_20260210122917/十万个梗库之：原神nb #原神 #抽象 _1.txt...
Copying gs://xhs-humor-data/extracted_knowledge/小红书博主【十万个梗库】笔记批量下载_20260210122917/十万个梗库之：四川婆罗门口音 #方言 #_1.txt...
Copying gs://xhs-humor-data/extracted_knowledge/小红书博主【十万个梗库】笔记批量下载_20260210122917/十万个梗库之：对齐颗粒度 #打工人 #职_1.txt...
Copying gs://xhs-humor-data/extracted_knowledge/小红书博主【十万个梗库】笔记批量下载_20260210122917/十万个大事件之：宗门大会 #蔡徐坤 #表_1.txt...
Copying gs://xhs-humor-data/extracted_knowledge/小红书博主【十万个梗库】笔记批量下载_20260210122917/十万个梗库之：次世代老登 #游戏_1.txt...
Copying gs://xhs-humor-data/extracted_knowledge/小红书博主【十万个梗库】笔记批量下载_20260210122917/十万个大事件之：短视频减速带 #影视 #_1.txt...
Copying gs://xhs-humor-data/extracted_knowledge/小红书博主【十万个梗库】笔记批量下载_20260210122917/十万个大事件之：川渝人均古风 #川渝_1.txt...
Copying gs://xhs-h